# ML Model Retraining — CI Sync
Retrains issue classifier & repairability regressor so `.joblib` files match the CI numpy/sklearn versions.

## Environment

In [ ]:
import numpy, sklearn, joblib, sys
print(f'numpy={numpy.__version__}')
print(f'sklearn={sklearn.__version__}')
print(f'joblib={joblib.__version__}')
print(f'python={sys.version.split()[0]}')

## 1. Issue / Damage Classifier

In [ ]:
import re, time, json, joblib, numpy as np, pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import LinearSVC

BASE_DIR = Path('.').resolve().parent
DATA_DIR = BASE_DIR / 'datasets'
MODEL_DIR = BASE_DIR / 'models'

def clean_text(value):
    if pd.isna(value): return ''
    text = str(value).lower()
    text = re.sub(r'[^a-z0-9]+', ' ', text.replace('\n', ' '))
    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
issue_2k = pd.read_csv(DATA_DIR / 'device_issue_dataset_2000.csv')
issue_2k = issue_2k.rename(columns={'issue': 'text', 'label': 'issue_label'})
issue_2k['source'] = 'device_issue_dataset'
issue_2k['text_clean'] = issue_2k['text'].apply(clean_text)
issue_2k['issue_label'] = issue_2k['issue_label'].fillna('Unknown').astype(str).str.strip()

final_df = pd.read_csv(DATA_DIR / 'final_dataset.csv')
final_df = final_df.rename(columns={'issue': 'text', 'predicted_label': 'issue_label'})
final_df['source'] = 'final_dataset'
final_df['text_clean'] = final_df['text'].apply(clean_text)
final_df['issue_label'] = final_df['issue_label'].fillna('Unknown').astype(str).str.strip()

issue_df = pd.concat([issue_2k, final_df], ignore_index=True)
issue_df = issue_df[['text', 'text_clean', 'issue_label', 'source']].dropna(subset=['text_clean'])
issue_df = issue_df[issue_df['issue_label'].astype(str).str.strip() != '']
print(f'Dataset: {len(issue_df)} rows, {issue_df["issue_label"].nunique()} labels')
print(f'Labels: {sorted(issue_df["issue_label"].unique())}')
issue_df.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    issue_df[['text_clean', 'source']], issue_df['issue_label'],
    test_size=0.2, random_state=42, stratify=issue_df['issue_label'],
)

pipeline = Pipeline([
    ('preprocess', ColumnTransformer([
        ('text', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200), 'text_clean'),
        ('source', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['source']),
    ], remainder='drop')),
    ('model', VotingClassifier(
        estimators=[
            ('svc', LinearSVC(C=1.0, class_weight='balanced', max_iter=1000)),
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42,
                                          class_weight='balanced_subsample', n_jobs=-1)),
            ('logreg', LogisticRegression(max_iter=2000, class_weight='balanced')),
        ],
        voting='hard',
    )),
])

t0 = time.time()
pipeline.fit(X_train, y_train)
elapsed = time.time() - t0
pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f'Trained in {elapsed:.1f}s — Accuracy: {acc:.4f}')

In [ ]:
print(classification_report(y_test, pred))

In [ ]:
from IPython.display import Image, display
display(Image(filename='output/issue_confusion_matrix.png'))

In [ ]:
joblib.dump(pipeline, MODEL_DIR / 'issue_classifier_voting.joblib')
print(f'Saved issue_classifier_voting.joblib')

## 2. Repairability Scoring Regressor

In [ ]:
from sklearn.ensemble import VotingRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score

repair_final = pd.read_csv(DATA_DIR / 'repairability-final.csv')
repair_final = repair_final.rename(columns={
    'Brand': 'brand', 'Model': 'model', 'Category': 'category',
    'Vision Score': 'repairability_score',
})
repair_final['source'] = 'repairability-final'
repair_final['device_text'] = repair_final[['brand', 'model', 'category']].fillna('').astype(str).agg(lambda r: ' '.join(r), axis=1)
repair_final['repairability_score'] = pd.to_numeric(repair_final['repairability_score'], errors='coerce')

ifixit = pd.read_csv(DATA_DIR / 'iFixit Repairability Scores (Public) - Smartphones.csv')
ifixit = ifixit.rename(columns={'OEM': 'brand', 'Device': 'model', 'Score': 'repairability_score'})
ifixit['category'] = 'Smartphone'
ifixit['source'] = 'ifixit'
ifixit['device_text'] = ifixit[['brand', 'model', 'category']].fillna('').astype(str).agg(lambda r: ' '.join(r), axis=1)
ifixit['repairability_score'] = pd.to_numeric(ifixit['repairability_score'], errors='coerce')

history = pd.read_csv(DATA_DIR / 'Repair-History.csv')
history = history.rename(columns={
    'Name': 'brand', 'Type': 'category', 'Problem': 'problem',
    'fixed YES/NO': 'fixed_flag',
})
history['source'] = 'repair-history'
history['device_text'] = history[['brand', 'category', 'problem']].fillna('').astype(str).agg(lambda r: ' '.join(r), axis=1)
history['repairability_score'] = history['fixed_flag'].astype(str).str.upper().eq('YES').astype(float) * 8.0 + 2.0

tech = pd.read_csv(DATA_DIR / 'tech_gadget_failures.csv')
tech = tech.rename(columns={
    'Device_Type': 'category', 'Brand': 'brand', 'Model_Name': 'model',
    'Failure_Type': 'failure_type', 'Repair_Cost': 'repair_cost',
    'Customer_Rating': 'customer_rating', 'Comments': 'comments',
    'Usage_Duration': 'usage_duration', 'Warranty_Status': 'warranty_status',
})
tech['source'] = 'tech-gadget-failures'
tech['device_text'] = tech[['brand', 'model', 'category', 'failure_type', 'comments']].fillna('').astype(str).agg(lambda r: ' '.join(r), axis=1)
tech['repairability_score'] = (
    10.0
    - (tech['repair_cost'].fillna(0) / 1200.0)
    - (tech['customer_rating'].fillna(5) < 3).astype(int) * 1.4
    - (tech['warranty_status'].astype(str).str.upper().eq('NO')).astype(int) * 0.5
).clip(1.0, 10.0)

laptop = pd.read_csv(DATA_DIR / 'laptop_dataset_final.csv', low_memory=False)
laptop = laptop.rename(columns={'Brand': 'brand', 'Model': 'model', 'Series': 'series', 'Price (Rs)': 'price'})
laptop['source'] = 'laptop-dataset'
laptop['category'] = 'Laptop'
laptop['device_text'] = laptop[['brand', 'model', 'series', 'category']].fillna('').astype(str).agg(lambda r: ' '.join(r), axis=1)
laptop['repairability_score'] = (5.0 + (pd.to_numeric(laptop['price'], errors='coerce').fillna(0) / 20000.0)).clip(1.0, 10.0)
laptop = laptop.sample(min(2000, len(laptop)), random_state=42)

combined = pd.concat([repair_final, ifixit, history, tech.sample(min(3000, len(tech)), random_state=42), laptop], ignore_index=True)
combined['repairability_score'] = pd.to_numeric(combined['repairability_score'], errors='coerce')
combined['device_text_clean'] = combined['device_text'].apply(clean_text)
combined = combined.dropna(subset=['repairability_score'])
print(f'Repairability dataset: {len(combined)} rows')

In [ ]:
model_df = combined.copy()
model_df['repair_cost'] = pd.to_numeric(model_df.get('repair_cost', 0), errors='coerce').fillna(0)
model_df['customer_rating'] = pd.to_numeric(model_df.get('customer_rating', 0), errors='coerce').fillna(0)
model_df['usage_duration'] = pd.to_numeric(model_df.get('usage_duration', 0), errors='coerce').fillna(0)

X = model_df[['device_text_clean', 'source', 'repair_cost', 'customer_rating', 'usage_duration']]
y = model_df['repairability_score']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

preprocessor = ColumnTransformer(transformers=[
    ('text', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200), 'device_text_clean'),
    ('num', 'passthrough', ['repair_cost', 'customer_rating', 'usage_duration']),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['source']),
], remainder='drop')

rpipe = Pipeline([
    ('preprocess', preprocessor),
    ('model', VotingRegressor(estimators=[
        ('tree', DecisionTreeRegressor(max_depth=6, random_state=42)),
        ('forest', RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)),
    ])),
])

t0 = time.time()
rpipe.fit(X_train, y_train)
elapsed = time.time() - t0
rep_pred = rpipe.predict(X_test)
mae = mean_absolute_error(y_test, rep_pred)
r2 = r2_score(y_test, rep_pred)
print(f'Trained in {elapsed:.1f}s — R²: {r2:.4f}  MAE: {mae:.4f}')

In [ ]:
from IPython.display import Image, display
display(Image(filename='output/repairability_metrics.png'))

In [ ]:
joblib.dump(rpipe, MODEL_DIR / 'repairability_voting_regressor.joblib')
print(f'Saved repairability_voting_regressor.joblib')

## 3. Summary

In [ ]:
summary = {
    'issue_model': {'accuracy': round(float(acc), 4), 'sample_count': int(len(issue_df))},
    'repairability_model': {'r2': round(float(r2), 4), 'mae': round(float(mae), 4), 'sample_count': int(len(combined))},
    'versions': {
        'numpy': numpy.__version__,
        'sklearn': sklearn.__version__,
        'joblib': joblib.__version__,
    },
}
print(json.dumps(summary, indent=2))
(MODEL_DIR / 'training_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Saved training_summary.json')